**ФИО студента:** [Иванов Иван Иванович]

**Задание:** `inpainting`

**Группа:** [номер группы]

**Дата сдачи:** [дата]


## Блок настройки окружения

Код для импорта библиотек, для формирования глобальных констант, других конфигураций (если требуется). Постарайтесь здесь зафиксировать все seed, которые сможете

In [ ]:
import random

import hashlib
from pathlib import Path
import time
import math

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image, display
from torch.utils.data import DataLoader
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda" and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT_NAME = "inpainting"
MLFLOW_MODEL_NAME = "inpainting_unet"

try:
    import mlflow
    from mlflow.exceptions import MlflowException
    from mlflow.tracking import MlflowClient

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
    MLFLOW_AVAILABLE = True
except Exception:
    MLFLOW_AVAILABLE = False

print(f"Трекинг MLflow: {'включен' if MLFLOW_AVAILABLE else 'отключен'}")


In [ ]:
device

## Блок "Данные"

Входной файл проверяется по SHA-256, форме и заполнению области пропусков нулями. После проверки куб транспонируется из `(x, y, z)` в `(y, z, x)` с формой `951×462×580` и предзагружается в оперативную память.

Пять соседних y-срезов дают 2.5D-контекст. Патч имеет форму `144×304`, целевая маска — `103×213`. Реальная область на каждом y-срезе имеет форму `103×580` и восстанавливается тремя позициями по `x`: `0:213`, `213:426`, `426:580`. Предыдущие срезы остаются известными, текущий срез маскируется от начала текущей полосы до правого края патча, будущие срезы маскируются внутри всей синтетической дыры. На последовательном инференсе предыдущие срезы и левая часть текущего среза содержат уже восстановленные значения.

Для train динамически генерируются положение патча по `z`, позиция полосы по `x`, положение маски, отражение по `z` и масштаб амплитуд. Отражение по `x` не используется, поскольку оно меняет направление последовательного восстановления. `TRAIN_SAMPLES_PER_CENTER` задаёт число примеров на каждый центральный y-срез за эпоху. Gaussian noise заранее записан в банк из 64 вариантов. Validation и test фиксированы и не аугментируются.

Held-out блок `HELD_OUT_BOX = (76, 179, 0, 580)` (`z=76:179`, вся ось `x`) исключён из train. Обучающие патчи располагаются в `z=282:462` и не пересекают held-out или реальный пропуск `z=179:282`. Validation и test строятся на held-out блоке с той же последовательной маскировкой. Их y-срезы разнесены защитными интервалами. Нормализация рассчитывается только по train-срезам и без held-out блока.


In [ ]:
from seismic_dataset import (
    HELD_OUT_BOX,
    PATCH_REGIONS,
    AMP_SCALE_RANGE,
    MASK_H,
    MASK_W,
    NOISE_SIGMA,
    NOISE_BANK_SIZE,
    PATCH_H,
    PATCH_W
)

DATA_PATH = Path("input_seismic_data/inpainting_cube_missed_data.npy")
EXPECTED_SHAPE = (580, 951, 462)
EXPECTED_SHA256 = "1aa956cfb12a75536ddd815f5d6f21abfbcccf474aec6de0d0a04c76b6eab1e5"

sha256 = hashlib.sha256()
with DATA_PATH.open("rb") as file:
    for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()
assert actual_sha256 == EXPECTED_SHA256, (
    f"SHA-256 не совпадает: ожидался {actual_sha256}, получен {actual_sha256}"
)

cube = np.load(DATA_PATH)
assert cube.shape == EXPECTED_SHAPE
assert cube.dtype == np.float32

real_mask = np.zeros(cube.shape, dtype=bool)
real_mask[:, 368:581, 179:282] = True

for slice_index in range(cube.shape[0]):
    assert np.all(cube[slice_index][real_mask[slice_index]] == 0), (
        f"Область пропусков в срезе {slice_index} содержит ненулевые значения"
    )

missing_count = int(real_mask.sum())
print(
    f"Проверка нулей пройдена: {missing_count:,} пропущенных значений, "
    f"{100 * missing_count / cube.size:.2f}% куба"
)

cube = np.ascontiguousarray(cube.transpose(1, 2, 0))
real_mask = np.ascontiguousarray(real_mask.transpose(1, 2, 0))
assert cube.shape == (951, 462, 580)
assert real_mask[368:581, 179:282, :].all()
print(f"Рабочая ориентация (y, z, x): {cube.shape}")

held_out_mask = np.zeros(cube.shape[1:], dtype=bool)
hz0, hz1, hx0, hx1 = HELD_OUT_BOX
held_out_mask[hz0:hz1, hx0:hx1] = True
assert not np.any(held_out_mask & real_mask.any(axis=0))

for z_min, z_max, x_min, x_max in PATCH_REGIONS:
    assert z_max - z_min >= PATCH_H
    assert x_max - x_min >= PATCH_W
assert PATCH_H % 16 == 0 and PATCH_W % 16 == 0
assert MASK_H + 32 <= PATCH_H and MASK_W + 32 <= PATCH_W


In [ ]:
from seismic_dataset import (
    CONTEXT_X,
    STRIP_STARTS,
    InpaintingDataset,
    make_inpainting_sample,
    overlaps_held_out,
    overlaps_real_hole,
    seed_worker
)

TRAIN_RATIO = 0.88
VAL_RATIO = 0.08
TEST_RATIO = 0.04
SPLIT_GAP = 4
SLICE_RANGE = (2, cube.shape[0] - 2)
TRAIN_SAMPLES_PER_CENTER = 7
VAL_SAMPLES_PER_CENTER = 8
TEST_SAMPLES_PER_CENTER = 4

ratios = np.array([TRAIN_RATIO, VAL_RATIO, TEST_RATIO])
ratios /= ratios.sum()
span = SLICE_RANGE[1] - SLICE_RANGE[0] - 2 * SPLIT_GAP
n_train = int(round(span * ratios[0]))
n_val = int(round(span * ratios[1]))
n_test = span - n_train - n_val
assert min(n_train, n_val, n_test) > 0

train_start = SLICE_RANGE[0]
val_start = train_start + n_train + SPLIT_GAP
test_start = val_start + n_val + SPLIT_GAP

train_centers = np.arange(train_start, train_start + n_train)
val_centers = np.arange(val_start, val_start + n_val)
test_centers = np.arange(test_start, test_start + n_test)


In [ ]:
train_sample_centers = train_centers[::4]
sample = cube[train_sample_centers, ::4, ::4]
sample_mask = real_mask[train_sample_centers, ::4, ::4]
sample_known = ~sample_mask & ~held_out_mask[::4, ::4][None]
mean = float(sample[sample_known].mean())
std = max(float(sample[sample_known].std()), 1e-6)

train_dataset = InpaintingDataset(
    cube, train_centers, TRAIN_SAMPLES_PER_CENTER, mean, std, True, 42, True
)
val_dataset = InpaintingDataset(
    cube, val_centers, VAL_SAMPLES_PER_CENTER, mean, std, False, 43, False, held_out=True
)
test_dataset = InpaintingDataset(
    cube, test_centers, TEST_SAMPLES_PER_CENTER, mean, std, False, 44, False, held_out=True
)

check_rng = np.random.default_rng(0)
for _ in range(2000):
    _, z, strip_index, *_ = make_inpainting_sample(check_rng, 0, True)
    x = STRIP_STARTS[strip_index] - CONTEXT_X
    assert not overlaps_held_out(z, x)
    assert not overlaps_real_hole(z)

print(f"train: {len(train_centers)} центров, {len(train_dataset)} примеров")
print(f"val: {len(val_centers)} центров, {len(val_dataset)} примеров")
print(f"test: {len(test_centers)} центров, {len(test_dataset)} примеров")


In [ ]:
BATCH_SIZE = 24
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0
)


## Блок "Модель"

Используется 2.5D U-Net с `base_channels = 24`: пять соседних срезов и маска подаются как шесть каналов. Bottleneck содержит свёртки с dilation 2 и 4. GroupNorm работает со всеми размерами каналов.

Все блоки внимания заменены на универсальный `TokenAttention` с размерностью токена 128 и четырьмя головами. Токены создаются через `Conv2d`, восстановление пространственной карты — через `ConvTranspose2d`. На 1/16 используется полное внимание по 171 токену. На 1/8, 1/4 и 1/2 используются 171, 684 и 2736 токенов; query создаются только для центральных 20% пропуска (3, 15 и 55 query), которые обращаются ко всем key/value своего масштаба. Крайние токены не участвуют в вычислении attention и не получают обновлений. Выходная проекция инициализирована нулями, поэтому в начале обучения каждый блок является identity-отображением.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


class DilatedBottleneck(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=2,
                dilation=2,
                bias=False
            ),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=4,
                dilation=4,
                bias=False
            ),
            nn.GroupNorm(8, out_channels),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


class TokenAttention(nn.Module):
    def __init__(
        self,
        channels,
        patch,
        token_dim=128,
        heads=4,
        mask_size=None,
        mask_stride=1,
        center_ratio=0.2
    ):
        super().__init__()
        assert token_dim % heads == 0
        self.patch = patch
        self.heads = heads
        self.head_dim = token_dim // heads
        self.full = mask_size is None
        self.to_tokens = nn.Conv2d(channels, token_dim, patch, patch, bias=False)
        self.pos = nn.Conv2d(token_dim, token_dim, 3, padding=1, groups=token_dim)
        self.norm = nn.LayerNorm(token_dim)
        self.qkv = nn.Linear(token_dim, 3 * token_dim, bias=False)
        self.out_proj = nn.Linear(token_dim, token_dim, bias=False)
        self.from_tokens = nn.ConvTranspose2d(token_dim, channels, patch, patch, bias=False)
        nn.init.zeros_(self.out_proj.weight)
        if not self.full:
            mask_h = -(-mask_size[0] // mask_stride)
            mask_w = -(-mask_size[1] // mask_stride)
            self.query_h = max(1, round(mask_h * center_ratio))
            self.query_w = max(1, round(mask_w * center_ratio))
            self.register_buffer("query_y", torch.arange(self.query_h), persistent=False)
            self.register_buffer("query_x", torch.arange(self.query_w), persistent=False)

    def forward(self, x, mask):
        b, _, h, w = x.shape
        tokens = self.to_tokens(x)
        th, tw = tokens.shape[-2:]
        tokens = tokens + self.pos(tokens)
        tokens = tokens.flatten(2).transpose(1, 2)
        qkv = self.qkv(self.norm(tokens)).view(
            b, th * tw, 3, self.heads, self.head_dim
        )
        q, k, v = (value.transpose(1, 2) for value in qkv.unbind(2))
        if self.full:
            out = F.scaled_dot_product_attention(q, k, v)
        else:
            mask_tokens = F.max_pool2d(mask, self.patch, self.patch)
            miss = mask_tokens[:, 0] > 0.5
            valid = miss.flatten(1).any(1)
            top = miss.any(dim=2).long().argmax(dim=1)
            left = miss.any(dim=1).long().argmax(dim=1)
            bottom = th - miss.flip(1).any(dim=2).long().argmax(dim=1)
            right = tw - miss.flip(2).any(dim=1).long().argmax(dim=1)
            y0 = top + ((bottom - top - self.query_h) // 2).clamp_min(0)
            x0 = left + ((right - left - self.query_w) // 2).clamp_min(0)
            query_y = y0[:, None] + self.query_y[None]
            query_x = x0[:, None] + self.query_x[None]
            indices = (
                query_y[:, :, None] * tw + query_x[:, None, :]
            ).flatten(1)
            gather_indices = indices[:, None, :, None].expand(
                -1, self.heads, -1, self.head_dim
            )
            q = torch.gather(q, 2, gather_indices)
            selected = F.scaled_dot_product_attention(q, k, v)
            selected = selected * valid[:, None, None, None]
            out = selected.new_zeros(b, self.heads, th * tw, self.head_dim)
            out = out.scatter(2, gather_indices, selected)
        out = out.transpose(1, 2).reshape(b, th * tw, -1)
        out = self.out_proj(out).transpose(1, 2).view(b, -1, th, tw)
        return x + self.from_tokens(out)


In [ ]:
class UNet25D(nn.Module):
    def __init__(
        self,
        in_channels=6,
        out_channels=1,
        base_channels=24,
        mask_size=(103, 213)
    ):
        super().__init__()
        self.pool = nn.AvgPool2d(2)
        self.encoders = nn.ModuleList([
            ConvBlock(in_channels, base_channels),
            ConvBlock(base_channels, base_channels * 2),
            ConvBlock(base_channels * 2, base_channels * 4),
            ConvBlock(base_channels * 4, base_channels * 8)
        ])
        self.bottleneck = DilatedBottleneck(base_channels * 8, base_channels * 16)
        self.upconvs = nn.ModuleList([
            nn.Conv2d(base_channels * 16, base_channels * 8, 3, padding=1),
            nn.Conv2d(base_channels * 8, base_channels * 4, 3, padding=1),
            nn.Conv2d(base_channels * 4, base_channels * 2, 3, padding=1),
            nn.Conv2d(base_channels * 2, base_channels, 3, padding=1)
        ])
        self.decoders = nn.ModuleList([
            ConvBlock(base_channels * 16, base_channels * 8),
            ConvBlock(base_channels * 8, base_channels * 4),
            ConvBlock(base_channels * 4, base_channels * 2),
            ConvBlock(base_channels * 2, base_channels)
        ])
        self.token_attn_16 = TokenAttention(base_channels * 16, patch=1)
        self.decoder_attentions = nn.ModuleList([
            TokenAttention(
                base_channels * 8,
                patch=2,
                mask_size=mask_size,
                mask_stride=16
            ),
            TokenAttention(
                base_channels * 4,
                patch=2,
                mask_size=mask_size,
                mask_stride=8
            ),
            TokenAttention(
                base_channels * 2,
                patch=2,
                mask_size=mask_size,
                mask_stride=4
            )
        ])
        self.head = nn.Conv2d(base_channels, out_channels, 1)

    def forward(self, x):
        height, width = x.shape[-2:]
        pad_height = (-height) % 16
        pad_width = (-width) % 16
        x = F.pad(x, (0, pad_width, 0, pad_height))
        mask = x[:, 5:6]

        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        x = self.token_attn_16(x, F.max_pool2d(mask, 16, 16))

        skips = list(reversed(skips))
        for index, (upconv, decoder, skip) in enumerate(zip(
            self.upconvs[:3],
            self.decoders[:3],
            skips[:3]
        )):
            x = F.interpolate(x, scale_factor=2, mode="nearest")
            x = upconv(x)
            x = torch.cat((x, skip), dim=1)
            x = decoder(x)
            scale = 2 ** (3 - index)
            x = self.decoder_attentions[index](
                x, F.max_pool2d(mask, scale, scale)
            )

        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.upconvs[3](x)
        x = torch.cat((x, skips[3]), dim=1)
        x = self.decoders[3](x)
        x = self.head(x)
        return x[..., :height, :width]


model = UNet25D()
print(f"Параметров модели: {sum(p.numel() for p in model.parameters()):,}")


## Блок "Обучение модели"

Потеря объединяет маскированные L1, MSE, gradient L1, focal frequency loss (FFL) по высоким частотам и `0.5 · (1 − R)`. FFL считается по спектру `rfft2((pred − target) · mask, norm="ortho")` в float32; низкие частоты ниже `FFL_HF_CUTOFF = 0.25` обнуляются, focal-вес `|Δ| / max|Δ|` считается внутри высокочастотной полосы. `FFL_WEIGHT = 0.05` линейно растёт за `FFL_WARMUP_EPOCHS = 5`; на test применяется полный вес. Норма градиентов не вычисляется и не логируется. Лучшая модель выбирается по глобальному validation R.

Основной запуск: `L1_WEIGHT = 0.3`, `GRAD_LOSS_WEIGHT = 0.3`, `FFL_WEIGHT = 0.05`, `R_LOSS_WEIGHT = 0.5`, `RUN_NAME = "sequential-y-x-causal"`. Для ablation можно установить `GRAD_LOSS_WEIGHT = 0.0` и другое имя запуска.

Используются `BATCH_SIZE = 24`, `EPOCHS = 100`, начальный `BASE_LR = 1e-3`. `ReduceLROnPlateau` работает в режиме `max` по validation R: при отсутствии улучшения уменьшает LR с `factor=0.5`, `patience=4`, до `MIN_LR = 1e-5`. Восстановление чекпоинта включает scheduler, early stopping и состояния генераторов случайных чисел.


In [ ]:
def masked_mse(prediction, target, mask):
    error = (prediction - target) ** 2
    return (error * mask).sum() / mask.sum().clamp_min(1)


def masked_l1(prediction, target, mask):
    error = torch.abs(prediction - target)
    return (error * mask).sum() / mask.sum().clamp_min(1)


def masked_gradient_l1(prediction, target, mask):
    mask_x = mask[..., 1:] * mask[..., :-1]
    mask_y = mask[..., 1:, :] * mask[..., :-1, :]
    error_x = (prediction[..., 1:] - prediction[..., :-1] - target[..., 1:] + target[..., :-1]).abs()
    error_y = (
        prediction[..., 1:, :] - prediction[..., :-1, :]
        - target[..., 1:, :] + target[..., :-1, :]
    ).abs()
    return (
        (error_x * mask_x).sum() + (error_y * mask_y).sum()
    ) / (mask_x.sum() + mask_y.sum()).clamp_min(1)


def hf_frequency_weight(height, width, cutoff, device):
    freq_y = torch.fft.fftfreq(height, device=device)[:, None] * 2
    freq_x = torch.fft.rfftfreq(width, device=device)[None, :] * 2
    radius = torch.sqrt((freq_y ** 2 + freq_x ** 2) / 2)
    return ((radius - cutoff) / (1 - cutoff)).clamp(0, 1)


def masked_focal_frequency_loss(prediction, target, mask):
    with torch.autocast(device_type=prediction.device.type, enabled=False):
        error = ((prediction - target) * mask.float()).float()
        magnitude = torch.fft.rfft2(error, norm="ortho").abs()
        band = hf_frequency_weight(
            error.shape[-2],
            error.shape[-1],
            FFL_HF_CUTOFF,
            error.device
        )
        magnitude = magnitude * band
        max_magnitude = magnitude.amax(dim=(-2, -1), keepdim=True).clamp_min(1e-8)
        focal_weight = magnitude / max_magnitude
        return (focal_weight * magnitude.square()).mean()


def correlation_r(prediction, target):
    prediction = prediction - prediction.mean()
    target = target - target.mean()
    denominator = prediction.norm() * target.norm()
    return (prediction * target).sum() / denominator.clamp_min(1e-8)


def masked_r(prediction, target, mask):
    return correlation_r(
        prediction[mask.bool()].float(),
        target[mask.bool()].float(),
    )


def reconstruction_loss(
    prediction,
    target,
    mask,
    ffl_weight=None,
    return_components=False
):
    if ffl_weight is None:
        ffl_weight = FFL_WEIGHT
    l1_loss = masked_l1(prediction, target, mask)
    mse_loss = masked_mse(prediction, target, mask)
    r_loss = 1 - masked_r(prediction, target, mask)
    components = {
        "l1_loss": L1_WEIGHT * l1_loss,
        "mse_loss": mse_loss,
        "r_loss": R_LOSS_WEIGHT * r_loss,
        "gradient_loss": prediction.new_zeros(()),
        "ffl_loss": prediction.new_zeros(()),
    }
    loss = components["l1_loss"] + components["mse_loss"] + components["r_loss"]
    if GRAD_LOSS_WEIGHT > 0:
        components["gradient_loss"] = GRAD_LOSS_WEIGHT * masked_gradient_l1(
            prediction,
            target,
            mask
        )
        loss = loss + components["gradient_loss"]
    if ffl_weight > 0:
        components["ffl_loss"] = ffl_weight * masked_focal_frequency_loss(
            prediction,
            target,
            mask
        )
        loss = loss + components["ffl_loss"]
    if return_components:
        return loss, components
    return loss


def new_r_moments():
    return torch.zeros(6, device=device)


def add_r_moments(moments, prediction, target, mask):
    prediction = prediction.detach()[mask.bool()].float()
    target = target[mask.bool()].float()
    update = torch.stack(
        (
            prediction.new_full((), float(prediction.numel())),
            prediction.sum(),
            target.sum(),
            (prediction * prediction).sum(),
            (target * target).sum(),
            (prediction * target).sum()
        )
    )
    return moments + update


def r_from_moments(moments):
    moments = moments.reshape(6)
    n = moments[0]
    sum_x = moments[1]
    sum_y = moments[2]
    sum_xx = moments[3]
    sum_yy = moments[4]
    sum_xy = moments[5]
    cov = n * sum_xy - sum_x * sum_y
    var_x = (n * sum_xx - sum_x * sum_x).clamp_min(1e-8)
    var_y = (n * sum_yy - sum_y * sum_y).clamp_min(1e-8)
    return (cov / (var_x.sqrt() * var_y.sqrt())).item()


In [ ]:
%load_ext tensorboard
from torch.utils.tensorboard import SummaryWriter

model = UNet25D().to(device)
model = torch.compile(model)
raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
EPOCHS = 100
EARLY_STOPPING_PATIENCE = 24
VALIDATION_EVERY = 1
L1_WEIGHT = 0.3
R_LOSS_WEIGHT = 0.5
GRAD_LOSS_WEIGHT = 0.3
FFL_WEIGHT = 0.05
FFL_WARMUP_EPOCHS = 5
FFL_HF_CUTOFF = 0.25


def ffl_weight_for_epoch(epoch):
    if FFL_WARMUP_EPOCHS == 0:
        return FFL_WEIGHT
    return FFL_WEIGHT * min(epoch / FFL_WARMUP_EPOCHS, 1.0)


BASE_LR = 1e-3
MIN_LR = 1e-5


optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=4,
    min_lr=MIN_LR
)

RUN_NAME = "sequential-y-x-causal"
RESUME = False
CHECKPOINT_PATH = "best_inpainting_unet.pt"
RUN_STAMP = time.strftime("%Y-%m-%d_%H-%M-%S")
LOG_DIR = f"runs/inpainting/{RUN_STAMP}"
writer = SummaryWriter(LOG_DIR)
best_val_r = -float("inf")
start_epoch = 1
epochs_without_improvement = 0
history = {
    "train_loss": [],
    "val_loss": [],
    "train_r": [],
    "val_r": []
}

if RESUME:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=device,
        weights_only=False
    )
    raw_model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_r = checkpoint["best_val_r"]
    history = checkpoint["history"]
    epochs_without_improvement = checkpoint["epochs_without_improvement"]
    random.setstate(checkpoint["python_random_state"])
    np.random.set_state(checkpoint["numpy_random_state"])
    torch.set_rng_state(checkpoint["torch_random_state"])
    if torch.cuda.is_available() and checkpoint["cuda_random_state"] is not None:
        torch.cuda.set_rng_state_all(checkpoint["cuda_random_state"])

if MLFLOW_AVAILABLE:
    if mlflow.active_run() is not None:
        mlflow.end_run()
    mlflow.start_run(run_name=f"{RUN_NAME}-epoch-{start_epoch}")
    mlflow.log_params(
        {
            "seed": SEED,
            "orientation": "y-z-x",
            "sequential_context": "restored-past-and-left",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "validation_every": VALIDATION_EVERY,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "scheduler": "plateau-r",
            "base_lr": BASE_LR,
            "min_lr": MIN_LR,
            "plateau_factor": 0.5,
            "plateau_patience": 4,
            "l1_weight": L1_WEIGHT,
            "r_loss_weight": R_LOSS_WEIGHT,
            "grad_loss_weight": GRAD_LOSS_WEIGHT,
            "ffl_weight": FFL_WEIGHT,
            "ffl_warmup_epochs": FFL_WARMUP_EPOCHS,
            "ffl_hf_cutoff": FFL_HF_CUTOFF,
            "attention": "token-pyramid-center20",
            "token_dim": 128,
            "token_heads": 4,
            "center_ratio": 0.2,
            "held_out_box": str(HELD_OUT_BOX),
            "train_regions": str(PATCH_REGIONS),
            "base_channels": 24,
            "dynamic_train_sampling": train_dataset.dynamic,
            "train_samples_per_center": TRAIN_SAMPLES_PER_CENTER,
            "val_samples_per_center": VAL_SAMPLES_PER_CENTER,
            "test_samples_per_center": TEST_SAMPLES_PER_CENTER,
            "amp_scale_range": str(AMP_SCALE_RANGE),
            "noise_sigma": NOISE_SIGMA,
            "noise_bank_size": NOISE_BANK_SIZE,
            "train_ratio": TRAIN_RATIO,
            "val_ratio": VAL_RATIO,
            "test_ratio": TEST_RATIO,
            "split_gap": SPLIT_GAP,
            "patch": f"{PATCH_H}x{PATCH_W}",
            "mask": f"{MASK_H}x{MASK_W}",
            "param_count": sum(p.numel() for p in model.parameters()),
        }
    )
    mlflow.set_tag("model/architecture", str(model))
    mlflow.set_tag("log_dir", LOG_DIR)


In [ ]:
dummy_inputs = torch.zeros(BATCH_SIZE, 6, PATCH_H, PATCH_W, device=device)
dummy_inputs[:, 5] = 1

with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
    warmup_loss = model(dummy_inputs).pow(2).mean()
warmup_loss.backward()
if device.type == "cuda":
    torch.cuda.synchronize()


In [ ]:
def train_epoch(ffl_weight):
    model.train()
    total_loss = torch.zeros((), device=device)
    component_totals = torch.zeros(5, device=device)
    r_moments = new_r_moments()
    for inputs, target, mask in tqdm(train_loader, desc="train", leave=False):
        inputs = inputs.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
            prediction = model(inputs)
            loss, components = reconstruction_loss(
                prediction,
                target,
                mask,
                ffl_weight,
                return_components=True
            )

        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        total_loss += loss.detach().float() * batch_size
        component_values = torch.stack([
            value.detach().float() for value in components.values()
        ])
        component_totals += component_values * batch_size
        r_moments = add_r_moments(r_moments, prediction, target, mask)

    train_r = r_from_moments(r_moments)
    component_totals = component_totals / len(train_loader.dataset)
    loss_components = dict(zip(components, component_totals.cpu().tolist()))

    return (total_loss / len(train_loader.dataset)).item(), train_r, loss_components


In [ ]:
def validation_epoch(ffl_weight):
    model.eval()
    total_loss = torch.zeros((), device=device)
    component_totals = torch.zeros(5, device=device)
    r_moments = new_r_moments()
    with torch.inference_mode():
        for inputs, target, mask in tqdm(val_loader, desc="val", leave=False):
            inputs = inputs.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
                prediction = model(inputs)
                loss, components = reconstruction_loss(
                    prediction,
                    target,
                    mask,
                    ffl_weight,
                    return_components=True
                )

            batch_size = inputs.size(0)
            total_loss += loss.float() * batch_size
            component_values = torch.stack([
                value.detach().float() for value in components.values()
            ])
            component_totals += component_values * batch_size
            r_moments = add_r_moments(r_moments, prediction, target, mask)

    val_r = r_from_moments(r_moments)
    component_totals = component_totals / len(val_loader.dataset)
    loss_components = dict(zip(components, component_totals.cpu().tolist()))

    return (total_loss / len(val_loader.dataset)).item(), val_r, loss_components


In [ ]:
pbar = tqdm(range(start_epoch, EPOCHS + 1))
for epoch in pbar:
    ffl_weight = ffl_weight_for_epoch(epoch)
    train_loss, train_r, train_loss_components = train_epoch(ffl_weight)

    history["train_loss"].append(train_loss)
    history["train_r"].append(train_r)
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("R/train", train_r, epoch)
    writer.add_scalar("Learning rate", optimizer.param_groups[0]["lr"], epoch)
    writer.add_scalar("Weights/FFL", ffl_weight, epoch)
    writer.add_scalar("Epoch", epoch, epoch)

    if MLFLOW_AVAILABLE:
        mlflow.log_metric("lr", optimizer.param_groups[0]["lr"], step=epoch)
        mlflow.log_metric("ffl_weight", ffl_weight, step=epoch)
        mlflow.log_metrics(
            {f"train_{name}": value for name, value in train_loss_components.items()},
            step=epoch
        )

    postfix = {"train_loss": f"{train_loss:.4f}", "train_R": f"{train_r:.4f}"}

    if epoch % VALIDATION_EVERY == 0:
        val_loss, val_r, val_loss_components = validation_epoch(ffl_weight)
        scheduler.step(val_r)

        history["val_loss"].append(val_loss)
        history["val_r"].append(val_r)
        writer.add_scalar("Loss/validation", val_loss, epoch)
        writer.add_scalar("R/validation", val_r, epoch)

        if val_r > best_val_r:
            best_val_r = val_r
            epochs_without_improvement = 0
            torch.save(
                {
                    "model_state_dict": raw_model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "mean": mean,
                    "std": std,
                    "epoch": epoch,
                    "best_val_r": best_val_r,
                    "history": history,
                    "epochs_without_improvement": epochs_without_improvement,
                    "python_random_state": random.getstate(),
                    "numpy_random_state": np.random.get_state(),
                    "torch_random_state": torch.get_rng_state(),
                    "cuda_random_state": (
                        torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
                    ),
                    "config": {
                        "architecture": "UNet25D",
                        "orientation": "y-z-x",
                        "sequential_context": "restored-past-and-left",
                        "base_channels": 24,
                        "attention": "token-pyramid-center20",
                        "token_dim": 128,
                        "token_heads": 4,
                        "center_ratio": 0.2,
                        "patch": [PATCH_H, PATCH_W],
                        "held_out_box": list(HELD_OUT_BOX),
                        "epochs": EPOCHS,
                        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
                        "l1_weight": L1_WEIGHT,
                        "r_loss_weight": R_LOSS_WEIGHT,
                        "grad_loss_weight": GRAD_LOSS_WEIGHT,
                        "ffl_weight": FFL_WEIGHT,
                        "ffl_warmup_epochs": FFL_WARMUP_EPOCHS,
                        "ffl_hf_cutoff": FFL_HF_CUTOFF,
                        "base_lr": BASE_LR,
                        "min_lr": MIN_LR,
                        "plateau_factor": 0.5,
                        "plateau_patience": 4
                    }
                },
                CHECKPOINT_PATH
            )
        else:
            epochs_without_improvement += 1

        writer.add_scalar("R/best_validation", best_val_r, epoch)
        postfix["val_loss"] = f"{val_loss:.4f}"
        postfix["val_R"] = f"{val_r:.4f}"

        if MLFLOW_AVAILABLE:
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("train_r", train_r, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("val_r", val_r, step=epoch)
            mlflow.log_metric("best_val_r", best_val_r, step=epoch)
            mlflow.log_metrics(
                {f"val_{name}": value for name, value in val_loss_components.items()},
                step=epoch
            )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            pbar.set_postfix(postfix)
            break

    writer.flush()
    pbar.set_postfix(postfix)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)
raw_model.load_state_dict(checkpoint["model_state_dict"])
best_epoch = checkpoint["epoch"]
writer.close()


In [ ]:
train_epochs = range(1, len(history["train_loss"]) + 1)
val_epochs = np.arange(
    VALIDATION_EVERY,
    len(history["val_loss"]) * VALIDATION_EVERY + 1,
    VALIDATION_EVERY
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_epochs, history["train_loss"], label="Train")
axes[0].plot(val_epochs, history["val_loss"], label="Validation")
axes[0].set(title="Функция потерь", xlabel="Эпоха", ylabel="Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(train_epochs, history["train_r"], label="Train")
axes[1].plot(val_epochs, history["val_r"], label="Validation")
axes[1].set(title="Коэффициент корреляции", xlabel="Эпоха", ylabel="R")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig("inpainting_training_history.png", dpi=150, bbox_inches="tight")
display(Image("inpainting_training_history.png"))


## Блок "Тестирование модели"

Модель загружается из чекпоинта `best_inpainting_unet.pt` (лучшая по валидационному R) и оценивается на тестовой выборке: средняя ошибка, коэффициент R и примеры предсказаний. При активном MLflow-запуске тестовые метрики и модель записываются на трекер, модель регистрируется в Model Registry, после чего запуск закрывается.


In [ ]:
model = UNet25D().to(device)
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


In [ ]:
model.eval()
test_loss = torch.zeros((), device=device)
prediction_values = []
target_values = []

with torch.inference_mode():
    for inputs, target, mask in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
            prediction = model(inputs)
            loss = reconstruction_loss(prediction, target, mask)

        test_loss += loss.float() * inputs.size(0)
        prediction_values.append(prediction[mask.bool()].float().cpu())
        target_values.append(target[mask.bool()].float().cpu())

test_loss = (test_loss / len(test_loader.dataset)).item()
test_predictions = torch.cat(prediction_values)
test_targets = torch.cat(target_values)
test_r = correlation_r(test_predictions, test_targets).item()

best_train_r = history["train_r"][best_epoch - 1] if history["train_r"] else 0
best_val_r = history["val_r"][best_epoch // VALIDATION_EVERY - 1] if history["val_r"] else 0
print(f"Test loss: {test_loss:.6f} | Test R: {test_r:.4f}")
print(
    f"Итоговые R | train: {best_train_r:.4f} | "
    f"val: {best_val_r:.4f} | test: {test_r:.4f}"
)

test_writer = SummaryWriter(LOG_DIR)
test_writer.add_scalar("Loss/test", test_loss, best_epoch)
test_writer.add_scalar("R/test", test_r, best_epoch)
test_writer.close()

if MLFLOW_AVAILABLE and mlflow.active_run() is not None:
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_r", test_r)
    model_info = mlflow.pytorch.log_model(
        model,
        artifact_path=MLFLOW_MODEL_NAME,
        serialization_format="pickle",
    )
    try:
        MlflowClient().create_registered_model(MLFLOW_MODEL_NAME)
    except MlflowException:
        pass
    MlflowClient().create_model_version(
        name=MLFLOW_MODEL_NAME,
        source=model_info.model_uri,
        run_id=mlflow.active_run().info.run_id,
        tags={"best-val-r": str(best_val_r)},
    )
    mlflow.end_run()


In [ ]:
inputs, target, mask = next(iter(test_loader))
inputs = inputs.to(device)
target = target.to(device)
mask = mask.to(device)

with torch.inference_mode():
    with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
        prediction = model(inputs)

examples = min(3, inputs.size(0))
fig, axes = plt.subplots(examples, 3, figsize=(15, 5 * examples))
titles = ("Оригинал", "Восстановленный срез", "Абсолютная ошибка")

for index in range(examples):
    original = target[index, 0].cpu().numpy() * std + mean
    predicted = prediction[index, 0].float().cpu().numpy() * std + mean
    sample_mask = mask[index, 0].cpu().numpy().astype(bool)

    restored = original.copy()
    restored[sample_mask] = predicted[sample_mask]

    error = np.full_like(original, np.nan)
    error[sample_mask] = np.abs(predicted[sample_mask] - original[sample_mask])

    a_min, a_max = np.percentile(original, [1, 99])
    panels = (
        (original, "gray", a_min, a_max),
        (restored, "gray", a_min, a_max),
        (error, "hot", None, None),
    )
    for column, (data, colormap, vmin, vmax) in enumerate(panels):
        axis = axes[index, column]
        axis.imshow(data, cmap=colormap, vmin=vmin, vmax=vmax, aspect="auto")
        axis.set_xticks([])
        axis.set_yticks([])

    axes[index, 0].set_ylabel(f"Тестовый пример {index + 1}")
    if index == 0:
        for column, title in enumerate(titles):
            axes[0, column].set_title(title)

fig.tight_layout()
fig.savefig("inpainting_test_examples.png", dpi=150, bbox_inches="tight")
display(Image("inpainting_test_examples.png"))


## Блок "Инференса модели"

Куб используется в ориентации `(y, z, x)` с формой `951×462×580`. Реальный пропуск находится в `y=368:581`, `z=179:282` и занимает всю ось `x`. На каждом y-срезе восстанавливается область `103×580`.

Модель получает патч `144×304` с `z=159:303`. Целевая маска имеет размер `103×213`. По оси `x` используются три позиции: `0:213`, `213:426`, `426:580`. Последняя полоса имеет ширину 154. Начало патча по `x` находится на 45 пикселей левее восстанавливаемой полосы.

Для каждой позиции по `x` выполняются 213 последовательных шагов `y=368:581`. Затем обработка возвращается к `y=368` для следующей позиции. Каналы `-1` и `-2` содержат уже восстановленные срезы. На первом шаге это известные срезы `366` и `367`, на последнем известные будущие срезы `581` и `582` не маскируются. Левая восстановленная часть текущего среза также используется как контекст при переходе к следующей x-позиции.

Визуализация использует итоговый `restored_cube`, поэтому повторный инференс для отдельных срезов не выполняется. Перед сохранением куб транспонируется обратно к исходной форме `580×951×462`.


In [ ]:
from seismic_dataset import (
    CONTEXT_X,
    MASK_H,
    MASK_W,
    REAL_HOLE_Y,
    REAL_HOLE_Z,
    STRIP_STARTS,
    VOLUME_W,
    extract_sequential_frames,
    sequential_masks
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

hole_rect = np.zeros(cube.shape[1:], dtype=bool)
hole_rect[REAL_HOLE_Z[0]:REAL_HOLE_Z[1], :] = True
assert np.array_equal(hole_rect, real_mask[REAL_HOLE_Y[0]])
assert not real_mask[:REAL_HOLE_Y[0]].any()
assert not real_mask[REAL_HOLE_Y[1]:].any()

INFERENCE_Z = REAL_HOLE_Z[0] - (PATCH_H - MASK_H) // 2
INFERENCE_MASK_Z = REAL_HOLE_Z[0] - INFERENCE_Z
assert INFERENCE_Z >= 0 and INFERENCE_Z + PATCH_H <= cube.shape[1]


def predict_strip(restored, center, strip_index):
    patch_x = STRIP_STARTS[strip_index] - CONTEXT_X
    patch = extract_sequential_frames(restored, center, INFERENCE_Z, patch_x)
    _, current_mask, future_mask = sequential_masks(strip_index, INFERENCE_MASK_Z)
    empty_mask = np.zeros_like(current_mask)
    future_masks = [
        future_mask if REAL_HOLE_Y[0] <= index < REAL_HOLE_Y[1] else empty_mask
        for index in (center + 1, center + 2)
    ]
    frame_masks = np.stack((
        empty_mask,
        empty_mask,
        current_mask,
        *future_masks
    ))

    patch = ((patch - mean) / std).astype(np.float32)
    patch *= 1 - frame_masks
    inputs = np.concatenate((patch, current_mask[None]), axis=0)[None]
    inputs = torch.from_numpy(inputs).float().to(device)

    with torch.inference_mode():
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
            prediction = model(inputs)

    return prediction[0, 0].float().cpu().numpy() * std + mean


def restore_volume():
    restored = cube.copy()
    total = len(STRIP_STARTS) * (REAL_HOLE_Y[1] - REAL_HOLE_Y[0])
    with tqdm(total=total, desc="inference") as progress:
        for strip_index, x_start in enumerate(STRIP_STARTS):
            for center in range(*REAL_HOLE_Y):
                prediction = predict_strip(restored, center, strip_index)
                width = min(MASK_W, VOLUME_W - x_start)
                restored[
                    center,
                    REAL_HOLE_Z[0]:REAL_HOLE_Z[1],
                    x_start:x_start + width
                ] = prediction[
                    INFERENCE_MASK_Z:INFERENCE_MASK_Z + MASK_H,
                    CONTEXT_X:CONTEXT_X + width
                ]
                progress.update(1)
    return restored


restored_cube = restore_volume()
assert restored_cube.shape == cube.shape


In [ ]:
def slice_panels(index):
    restored = restored_cube[index]
    original = cube[index].copy()
    missing = real_mask[index]

    original_display = original.copy()
    original_display[missing] = np.nan

    restored_region = np.full_like(restored, np.nan)
    restored_region[missing] = restored[missing]

    known = original[~missing]
    a_min, a_max = np.percentile(known, [1, 99])
    return original_display, restored_region, restored, a_min, a_max


def show_restored_slice(index):
    original_display, restored_region, restored, a_min, a_max = slice_panels(index)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    panels = (
        (original_display, "Исходный срез"),
        (restored_region, "Восстановленная область"),
        (restored, "Результат"),
    )
    for column, (data, title) in enumerate(panels):
        axes[column].imshow(data, cmap="gray", vmin=a_min, vmax=a_max, aspect="auto")
        axes[column].set_title(title)
        axes[column].set_xticks([])
        axes[column].set_yticks([])

    fig.suptitle(f"Срез y={index}, ориентация (z, x)")
    fig.tight_layout()
    fig.savefig("inpainting_restored_slice.png", dpi=150, bbox_inches="tight")
    display(Image("inpainting_restored_slice.png"))


In [ ]:
indices = np.linspace(REAL_HOLE_Y[0], REAL_HOLE_Y[1] - 1, 4).astype(int)
panels = [slice_panels(index) for index in indices]

titles = ("Исходный срез", "Восстановленная область", "Результат")
fig, axes = plt.subplots(len(indices), 3, figsize=(15, 5 * len(indices)))

for row, index in enumerate(indices):
    original_display, restored_region, restored, a_min, a_max = panels[row]
    for column, (data, title) in enumerate(zip((original_display, restored_region, restored), titles)):
        axis = axes[row, column]
        axis.imshow(data, cmap="gray", vmin=a_min, vmax=a_max, aspect="auto")
        axis.set_xticks([])
        axis.set_yticks([])
    axes[row, 0].set_ylabel(f"y={index}")
    if row == 0:
        for column, title in enumerate(titles):
            axes[0, column].set_title(title)

fig.tight_layout()
fig.savefig("inpainting_restored_slices.png", dpi=150, bbox_inches="tight")
display(Image("inpainting_restored_slices.png"))


## Блок сохранения артефактов

Код и информация о том, как сохранялась модель и восстановленный куб (shape финального куба должен совпадать с shape'ом входного куба, который был скачан по ссылке)

In [ ]:
output_cube = np.ascontiguousarray(restored_cube.transpose(2, 0, 1))
assert output_cube.shape == EXPECTED_SHAPE

output_path = "inpainting_cube_restored.npy"
model_output_path = "inpainting_unet_inference.pt"
np.save(output_path, output_cube)
torch.save(model.state_dict(), model_output_path)
print(f"Восстановленный куб сохранен: {output_path}")
print(f"Модель для инференса сохранена: {model_output_path}")
